# Heretic Colab Setup
このノートブックは Heretic を Google Colab 上で動かします。GPU を選択し、必要な依存をインストールしてからモデルを実行します。

In [ ]:
#@title Optional: Mount Google Drive (チェックポイント保存のみ)
# Drive に依存したくない場合は False にしてください（デフォルト: False）。
MOUNT_DRIVE = False  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    import os
    print('Mounting Google Drive...')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/heretic'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive mounted at /content/drive. Using', DRIVE_ROOT)
else:
    print('Google Drive mounting skipped. Checkpoints will default to /content/checkpoints unless you change settings below.')


In [1]:
# ランタイム確認（GPU が利用可能か確認し、なければ対処法を表示します）
import shutil, sys, textwrap
if shutil.which('nvidia-smi') is None:
    print(textwrap.dedent('''
nvidia-smi が見つかりません。GPU が有効なランタイムを選択してください。
'''))
else:
    # GPU ドライバ情報を表示
    !nvidia-smi

Sat Apr 18 07:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# pip を最新化して依存を入れる（CUDA バージョンに合わせて index-url を変更してください）
!pip install pip -q
!pip install uv -q

!uv self version

uv 0.11.7 (x86_64-unknown-linux-gnu)


リポジトリをクローンします。

In [ ]:
#@title Clone Heretic (フォームで Repo/Branch を指定)
repo = "Akikukeo1/Live-Vision-Narrator"  #@param {type:"string"}
branch = "main"  #@param {type:"string"}
dest = "/content/Live-Vision-Narrator"  #@param {type:"string"}

!git clone --depth 1 --branch {branch} https://github.com/{repo}.git {dest}

# heretic サブディレクトリを指定
HERETIC_PATH = f"{dest}/heretic"
print(f'✓ HERETIC_PATH set to {HERETIC_PATH}')


依存をインストール・アップデートする必要があります。

In [ ]:
#@title Install heretic via uv (必須)
# NOTE: Clone セルを先に実行してください。

%%bash -s $HERETIC_PATH
cd $1

uv venv --clear
uv pip install -e .
uv pip install --upgrade transformers --no-cache
uv pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu130 --no-cache

echo ""
echo "✓ All dependencies installed and upgraded successfully."


In [ ]:
# チェックポイント保存先の設定（Drive をマウントしている場合は Drive に、そうでなければ /content に保存）
import os, shutil
# MOUNT_DRIVE が定義されていなければ False を仮定
try:
    MOUNT_DRIVE
except NameError:
    MOUNT_DRIVE = False

if MOUNT_DRIVE:
    CHECKPOINT_DIR = '/content/drive/MyDrive/heretic/checkpoints'
else:
    CHECKPOINT_DIR = '/content/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_DIR)

# 簡単な同期例（必要に応じて手動で実行）
print('To sync checkpoints to Drive later (if MOUNT_DRIVE is True):')
print('  !rsync -av --progress /content/checkpoints/ /content/drive/MyDrive/heretic/checkpoints/')
print('Or use copy:')
print('  !cp -r /content/checkpoints/* /content/drive/MyDrive/heretic/checkpoints/')
